# WET-007: WEP Performance TIE vs Danish

Owner: **John Franklin Crenshaw** <br>
Last Verified to Run: **2024-10-27** <br>
Software Version:
  - `ts_wep`: **12.4.1** (on branch `tickets/DM-47188`)
  - `lsst_distrib`: **w_2024_43**

## Test Description

This notebook addresses [SITCOM-1149](https://rubinobs.atlassian.net/browse/SITCOM-1149) by comparing performance of TIE vs Danish algorithms on donuts from 20241027.

Data for this analyis was created by running the following two commands:

TIE:
```sh
pipetask run -j 4 -b /repo/embargo -i LSSTComCam/defaults -o u/crenshaw/WET-007_tie_20241027_2 -p /home/c/crenshaw/notebooks/comcam_commissioning/prep_for_WET-007/pipeline_WET-007_tie.yaml -d "instrument='LSSTComCam' and exposure.observation_type='cwfs' and visit in (2024102700016,2024102700017)" --rebase --register-dataset-types
```

Danish:
```sh
pipetask run -j 4 -b /repo/embargo -i LSSTComCam/defaults -o u/crenshaw/WET-007_danish_20241027_2 -p /home/c/crenshaw/notebooks/comcam_commissioning/prep_for_WET-007/pipeline_WET-007_danish.yaml -d "instrument='LSSTComCam' and exposure.observation_type='cwfs' and visit in (2024102700016,2024102700017)" --rebase --register-dataset-types
```

where the contents of `pipeline_WET-007_tie.yaml` are
```yaml
description: Pipeline for WET-007 using TIE
instrument: lsst.obs.lsst.LsstComCam
tasks:
  generateDonutDirectDetectTask:
    class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
    config:
      donutSelector.useCustomMagLimit: True
      donutSelector.sourceLimit: 5
  cutOutDonutsScienceSensorGroupTask:
    class: lsst.ts.wep.task.cutOutDonutsScienceSensorTask.CutOutDonutsScienceSensorTask
    config:
      python: |
        from lsst.ts.wep.task.pairTask import GroupPairer
        config.pairer.retarget(GroupPairer)
      donutStampSize: 160
      initialCutoutPadding: 40
  calcZernikesTask:
    class: lsst.ts.wep.task.calcZernikesTask.CalcZernikesTask
    config:
      estimateZernikes.maxNollIndex: 28
      estimateZernikes.saveHistory: False
      estimateZernikes.maskKwargs: {'doMaskBlends': False}
  isr:
    class: lsst.ip.isr.IsrTaskLSST
    config:
      # Although we don't have to apply the amp offset corrections, we do want
      # to compute them for analyzeAmpOffsetMetadata to report on as metrics.
      doAmpOffset: true
      ampOffset.doApplyAmpOffset: false
      # Turn off slow steps in ISR
      doBrighterFatter: false
      # Mask saturated pixels,
      # but turn off quadratic crosstalk because it's currently broken
      doSaturation: True
      crosstalk.doQuadraticCrosstalkCorrection: False
```

and the contents of `pipeline_WET-007_danish.yaml` are
```yaml
description: Pipeline for WET-007 using TIE
instrument: lsst.obs.lsst.LsstComCam
tasks:
  generateDonutDirectDetectTask:
    class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
    config:
      donutSelector.useCustomMagLimit: True
      donutSelector.sourceLimit: 5
  cutOutDonutsScienceSensorGroupTask:
    class: lsst.ts.wep.task.cutOutDonutsScienceSensorTask.CutOutDonutsScienceSensorTask
    config:
      python: |
        from lsst.ts.wep.task.pairTask import GroupPairer
        config.pairer.retarget(GroupPairer)
      donutStampSize: 160
      initialCutoutPadding: 40
  calcZernikesTask:
    class: lsst.ts.wep.task.calcZernikesTask.CalcZernikesTask
    config:
      estimateZernikes.maxNollIndex: 28
      estimateZernikes.saveHistory: False
      python: |
        from lsst.ts.wep.task import EstimateZernikesDanishTask
        config.estimateZernikes.retarget(EstimateZernikesDanishTask)
  isr:
    class: lsst.ip.isr.IsrTaskLSST
    config:
      # Although we don't have to apply the amp offset corrections, we do want
      # to compute them for analyzeAmpOffsetMetadata to report on as metrics.
      doAmpOffset: true
      ampOffset.doApplyAmpOffset: false
      # Turn off slow steps in ISR
      doBrighterFatter: false
      # Mask saturated pixels,
      # but turn off quadratic crosstalk because it's currently broken
      doSaturation: True
      crosstalk.doQuadraticCrosstalkCorrection: False
```

# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler

In [ ]:
# Change this path to appropriate butler when on-sky images arrive
path_to_aos_butler = '/repo/embargo'
butler = Butler(path_to_aos_butler)

tie_collection = "u/crenshaw/WET-007_tie_20241027_2"
danish_collection = "u/crenshaw/WET-007_danish_20241027_2"

## Plot donut stamps

First we will plot the donut stamps to see what they look like.

Above each stamp I print the field angle of the donut so we can identify whether the TIE and Danish donut pairs are the same.

#### Notes & Questions:
- It looks like we need to expand the default stamp size
- It looks like the pairing is inconsistent between the TIE and Danish. *Why?*
- Some poorly interpolated donuts are making it through. I will investigate why.

In [ ]:
def plot_pairs(ID, collection, axes, title):
    # Determine if each pair was selected 
    quality = butler.get("donutQualityTable", dataId=ID, collections=collection)
    intraSelected = quality[quality["DEFOCAL_TYPE"] == "intra"]["FINAL_SELECT"]
    extraSelected = quality[quality["DEFOCAL_TYPE"] == "extra"]["FINAL_SELECT"]

    intraStamps = butler.get("donutStampsIntra", dataId=ID, collections=collection)
    intraStamps = [stamp for stamp, selected in zip(intraStamps, intraSelected) if selected]
    extraStamps = butler.get("donutStampsIntra", dataId=ID, collections=collection)
    extraStamps = [stamp for stamp, selected in zip(extraStamps, extraSelected) if selected]
    
    usedList = butler.get("zernikes", dataId=ID, collections=collection)[1:]["used"]
    
    for i, (used, intra, extra) in enumerate(zip(usedList, intraStamps, extraStamps)):
        axes[0, i].imshow(intra.wep_im.image, origin="lower")
        angle = intra.wep_im.fieldAngle
        axes[0, i].set_title(f"({angle[0]:.2f}, {angle[1]:.2f})", fontsize=8)
        
        axes[1, i].imshow(extra.wep_im.image, origin="lower")
        angle = extra.wep_im.fieldAngle
        axes[1, i].set_title(f"({angle[0]:.2f}, {angle[1]:.2f})", fontsize=8)
        if not used:
            npix = intra.wep_im.image.shape[0] - 1
            axes[0, i].plot([0, npix], [0, npix], c="r", lw=1)
            axes[0, i].plot([0, npix], [npix, 0], c="r", lw=1)
            axes[1, i].plot([0, npix], [0, npix], c="r", lw=1)
            axes[1, i].plot([0, npix], [npix, 0], c="r", lw=1)
    for ax in axes[:, len(usedList):].flatten():
        ax.set_axis_off()
    for ax in axes.flatten():
        ax.set(xticks=[], yticks=[])
    axes[0, 0].set_ylabel(f"{title}\nIntra")
    axes[1, 0].set_ylabel(f"{title}\nExtra")

In [ ]:
tie_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=tie_collection, datasets='donutStampsIntra'))
dan_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=danish_collection, datasets='donutStampsIntra'))

for t_id, d_id in zip(tie_ids, dan_ids):
    fig, axes = plt.subplots(4, 5, dpi=120, constrained_layout=True)
    plot_pairs(t_id, tie_collection, axes[:2], "TIE")
    plot_pairs(d_id, danish_collection, axes[2:4], "Danish")
    fig.suptitle(f"Detector {t_id['detector']}")

## Load Zernike Estimates

Below we load the Zernikes from the two pipelines above and compare the outputs from the TIE and Danish

In [ ]:
# Load zernike estimates
tie_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=tie_collection, datasets='zernikeEstimateAvg'))
tie_zks = []
for data_id in tie_ids:
    table = butler.get('zernikes', dataId=data_id, collections=tie_collection)
    tie_zks.append(np.array([table[table["label"] == "average"][f"Z{i}"][0].value for i in range(4, 29)]))
tie_zks = np.array(tie_zks)

dan_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=danish_collection, datasets='zernikeEstimateAvg'))
dan_zks = []
for data_id in tie_ids:
    table = butler.get('zernikes', dataId=data_id, collections=danish_collection)
    dan_zks.append(np.array([table[table["label"] == "average"][f"Z{i}"][0].value for i in range(4, 29)]))
dan_zks = np.array(dan_zks)

In [ ]:
fig, ax = plt.subplots(1, 1, dpi=150)

ax.errorbar(np.arange(4, 29), np.mean(tie_zks, axis=0), yerr=np.std(tie_zks, axis=0), ls="--", c="C2", label="TIE", capsize=4)
ax.errorbar(np.arange(4, 29), np.mean(dan_zks, axis=0), yerr=np.std(dan_zks, axis=0), ls="--", c="cornflowerblue", alpha=1, label="Danish", capsize=4)
ax.scatter([4, 5, 6, 7, 8], [-1573.3, 819.4, 1290.6, -1920.3, 671.3], c="r", marker="x", label="Josh", zorder=10, s=70)

ax.legend()

ax.set(
    xlabel="Noll index",
    ylabel="Amplitude [nm]",
    xticks=np.arange(4, 29, 2),
    xlim=(3.5, 28.5),
)

# inset Axes....
x1, x2, y1, y2 = 3.5, 8.75, -2300, +2500  # subregion of the original image
axins = ax.inset_axes(
    [0.5, 0.075, 0.4, 0.35],
    xlim=(x1, x2), ylim=(y1, y2), xticks=np.arange(4, 9), yticks=[-2e3, -1e3, 0, 1e3, 2e3], ylabel="nm")

axins.errorbar(np.arange(4, 29), np.mean(tie_zks, axis=0), yerr=np.std(tie_zks, axis=0), ls="--", c="C2", label="TIE", capsize=4)
axins.errorbar(np.arange(4, 29), np.mean(dan_zks, axis=0), yerr=np.std(dan_zks, axis=0), ls="--", c="cornflowerblue", alpha=1, label="Danish", capsize=4)
axins.scatter([4, 5, 6, 7, 8], [-1573.3, 819.4, 1290.6, -1920.3, 671.3], c="r", marker="x", label="Josh", zorder=10, s=70)


ax.indicate_inset_zoom(axins, edgecolor="gray")

plt.show()

The TIE and Danish agree within their purported uncertainties.
The 4 points labeled "Josh" were created manually by Josh Meyers, who used Danish to jointly fit a single intra- & extra-focal donut.

Note these means are the means of the combined Zernikes from each of the 9 detectors.
The uncertainties are the standard deviations of these combined Zernikes.